# Day 035 Project: Auto-Analyst — Your Daily Digest Pipeline

## What You're Building

A `DigestPipeline` that fetches content from at least 4 sources, extracts structured information, and produces a consolidated editorial digest.

## Project Requirements

1. Create a `DigestPipeline` instance stored as `pipeline`
2. Add at least 4 sources (inline text strings, file paths, or URLs)
3. Run the full pipeline with `await pipeline.process()`
4. Print the digest and at least one per-article summary
5. Check the `ok_count` and `source_count` from the result
6. Verify with `_run_project_checks()`

## Provided: All Implementations

In [ ]:
import requests
from pathlib import Path

def fetch_text(source: str) -> dict:
    src  = str(source)
    text = None
    kind = 'text'

    if src.startswith('http://') or src.startswith('https://'):
        try:
            response = requests.get(src, timeout=10)
            response.raise_for_status()
            text = response.text
            kind = 'url'
        except Exception as e:
            text = '[fetch error: ' + str(e) + ']'
            kind = 'url_error'
    else:
        try:
            p = Path(src)
            if p.exists() and p.is_file():
                text = p.read_text(encoding='utf-8')
                kind = 'file'
        except Exception:
            pass

    if text is None:
        text = src
        kind = 'text'

    return {
        'source':     src,
        'kind':       kind,
        'content':    text,
        'char_count': len(text),
    }


import json
import ollama
from pydantic import BaseModel, Field

class ArticleInfo(BaseModel):
    title:      str       = Field(description='Topic or title in 3-6 words')
    summary:    str       = Field(description='One sentence summary')
    sentiment:  str       = Field(description='positive, negative, or neutral')
    key_points: list[str] = Field(default_factory=list,
                                  description='Up to 3 key points as short phrases')

def extract_info(doc: dict, model: str = 'llama3.2') -> dict:
    schema = ArticleInfo.model_json_schema()
    prompt = (
        'Extract information from the document below. '
        'Return valid JSON matching this schema:\n'
        + json.dumps(schema, indent=2)
        + '\n\nDocument:\n' + doc['content'][:1500]
    )
    try:
        response = ollama.chat(
            model=model,
            messages=[{'role': 'user', 'content': prompt}],
            format='json',
        )
        info = ArticleInfo.model_validate_json(response['message']['content'])
        return {**doc, 'info': info.model_dump(), 'status': 'ok',    'error': None}
    except Exception as e:
        return {**doc, 'info': None,               'status': 'error', 'error': str(e)}


import asyncio
import json
import ollama

async def async_extract(doc: dict, model: str = 'llama3.2') -> dict:
    client = ollama.AsyncClient()
    schema = ArticleInfo.model_json_schema()
    prompt = (
        'Extract information from the document below. '
        'Return valid JSON matching this schema:\n'
        + json.dumps(schema, indent=2)
        + '\n\nDocument:\n' + doc['content'][:1500]
    )
    try:
        response = await client.chat(
            model=model,
            messages=[{'role': 'user', 'content': prompt}],
            format='json',
        )
        info = ArticleInfo.model_validate_json(response['message']['content'])
        return {**doc, 'info': info.model_dump(), 'status': 'ok',    'error': None}
    except Exception as e:
        return {**doc, 'info': None,               'status': 'error', 'error': str(e)}


async def batch_extract(docs: list, max_concurrent: int = 3,
                        model: str = 'llama3.2') -> list[dict]:
    if not docs:
        return []
    sem = asyncio.Semaphore(max_concurrent)
    async def _run(doc):
        async with sem:
            return await async_extract(doc, model)
    return list(await asyncio.gather(*[_run(d) for d in docs]))


import ollama

def generate_digest(results: list, model: str = 'llama3.2') -> str:
    ok     = [r for r in results if r.get('status') == 'ok']
    errors = [r for r in results if r.get('status') == 'error']
    if not ok:
        return 'No articles extracted successfully (' + str(len(errors)) + ' errors).'
    lines = [
        '=== Auto-Analyst Digest ===',
        str(len(results)) + ' sources processed: '
        + str(len(ok)) + ' ok, ' + str(len(errors)) + ' failed.\n',
    ]
    for i, r in enumerate(ok, 1):
        info      = r.get('info') or {}
        title     = info.get('title',     'Untitled')
        summary   = info.get('summary',   '')
        sentiment = info.get('sentiment', 'unknown')
        kp        = info.get('key_points', [])
        kp_text   = '; '.join(kp[:3]) if kp else ''
        lines.append('[' + str(i) + '] ' + title + '  [' + sentiment + ']')
        lines.append('    ' + summary)
        if kp_text:
            lines.append('    Key points: ' + kp_text)
        lines.append('')
    context  = '\n'.join(lines)
    prompt   = (
        context + '\n\n'
        'Write a 3-4 sentence editorial digest identifying '
        'the main themes, patterns, and key insights across all articles.'
    )
    response = ollama.chat(
        model=model,
        messages=[{'role': 'user', 'content': prompt}],
    )
    return response['message']['content']


import asyncio

class DigestPipeline:
    def __init__(self, model: str = 'llama3.2', max_concurrent: int = 3):
        self.model          = model
        self.max_concurrent = max_concurrent
        self._sources: list = []

    def add_source(self, source) -> 'DigestPipeline':
        self._sources.append(source)
        return self

    async def process(self) -> dict:
        docs     = [fetch_text(s) for s in self._sources]
        results  = await batch_extract(docs, self.max_concurrent, self.model)
        digest   = generate_digest(results, self.model)
        ok_count = sum(1 for r in results if r.get('status') == 'ok')
        return {
            'source_count': len(docs),
            'ok_count':     ok_count,
            'results':      results,
            'digest':       digest,
        }

    def run(self) -> dict:
        return asyncio.run(self.process())

## Your Pipeline

In [ ]:
pipeline = (
    DigestPipeline(model='llama3.2', max_concurrent=3)
    .add_source('Python is a high-level programming language popular in data science.')
    .add_source('Machine learning enables computers to learn from data patterns.')
    .add_source('Natural language processing lets computers read and generate text.')
    .add_source('Cloud computing provides on-demand access to computing resources.')
)

# TODO: run the pipeline and store the result
# output = await pipeline.process()

# TODO: print the digest
# print('\n=== DIGEST ===')
# print(output['digest'])

# TODO: print a summary of each result
# for r in output['results']:
#     info = r.get('info') or {}
#     print(f"  [{r['status']}] {info.get('title', 'N/A')}")

## Checks

In [ ]:
async def _run_project_checks():
    total = 5
    passed = 0

    # Check 1: pipeline is a DigestPipeline
    try:
        assert 'pipeline' in globals()
        assert isinstance(pipeline, DigestPipeline)
        passed += 1; print('\u2705 Check 1: pipeline is a DigestPipeline')
    except Exception as e:
        print(f'\u274c Check 1: {e}')

    # Check 2: at least 4 sources
    try:
        assert len(pipeline._sources) >= 4, \
            f'need >= 4 sources, got {len(pipeline._sources)}'
        passed += 1; print(f'\u2705 Check 2: {len(pipeline._sources)} sources registered')
    except Exception as e:
        print(f'\u274c Check 2: {e}')

    # Check 3: output is defined
    try:
        assert 'output' in globals(), \
            'output not defined — run: output = await pipeline.process()'
        assert isinstance(output, dict)
        for k in ('source_count', 'ok_count', 'results', 'digest'):
            assert k in output, f'output missing key: {k}'
        passed += 1; print('\u2705 Check 3: output has source_count, ok_count, results, digest')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: source_count >= 4 and digest is non-empty
    try:
        assert output['source_count'] >= 4, \
            f'source_count should be >= 4, got {output["source_count"]}'
        assert isinstance(output['digest'], str) and output['digest'].strip(), \
            'digest is empty'
        passed += 1; print(f'\u2705 Check 4: {output["source_count"]} sources, digest non-empty')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: at least some ok results
    try:
        ok = [r for r in output['results'] if r.get('status') == 'ok']
        assert ok, 'no successful extractions (is Ollama running?)'
        assert output['ok_count'] == len(ok), \
            f'ok_count mismatch: {output["ok_count"]} vs {len(ok)}'
        passed += 1; print(f'\u2705 Check 5: {len(ok)}/{output["source_count"]} extractions ok')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Project complete!')
    print(f'\nScore: {passed}/{total}')


await _run_project_checks()

## Bonus Challenges

- Swap inline text for real URLs (news sites, Wikipedia) and run with `fetch_text`
- Save `output` to a JSON file with `json.dumps(output, indent=2)`
- Add a CLI entry point using `AICli` from Day 34: `ai-digest --source 'text...' --source 'text...'`
- Schedule the pipeline with `APScheduler` from Day 27 for a real daily digest
- Add a Slack webhook delivery using Day 29's webhook pattern
- Use `SecureConfig` from Day 32 to load model name and max_concurrent from a `.env` file